### Code Description;
This code processes and saves data from four MMS satellites (MMS1, MMS2, MMS3, MMS4) for a specified time range (e.g. 2016-03-06/19:25:00 to 2016-03-06/19:30:00). It loads magnetic field, density, and position data from each satellite, interpolates these datasets to have the same time resolution, and then saves the processed data into CSV files in a folder named by the time range. The filenames reflect the time range and satellite number. This ensures that all data points across different variables are aligned in time, providing a consistent dataset for further analysis.

Date of coding: 2 Juy 2024 

### Time input

In [ ]:
# Define the start and end times for the data range
# From
t1 = '2016-03-06/0:00:00'

# to
t2 = '2016-03-06/23:59:59'

### Data aquisition, interpolation, and save CSV files

In [ ]:
import pyspedas
from pytplot import tplot
from pyspedas.mms import dsp
from pyspedas.mms import fsm
from pytplot import options
from pytplot import get_data
from pyspedas.mms.mms_orbit_plot import mms_orbit_plot
from pyspedas import mms_load_mec
from pyspedas.mms import mec
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from datetime import timedelta
import os

# Calculate the extended time range for interpolation (5 minutes before and after)
t1_ext = (pd.to_datetime(t1) - timedelta(minutes=5)).strftime('%Y-%m-%d/%H:%M:%S')
t2_ext = (pd.to_datetime(t2) + timedelta(minutes=5)).strftime('%Y-%m-%d/%H:%M:%S')

# List of MMS probes to process (MMS1, MMS2, MMS3, MMS4)
probes = [1]

# Ensure t1 and t2 are timezone-aware
t1_aware = pd.to_datetime(t1).tz_localize('UTC')
t2_aware = pd.to_datetime(t2).tz_localize('UTC')

# Create folder for saving CSV files
t1_formatted = t1.split("/")[1].replace(":", "_")
t2_formatted = t2.split("/")[1].replace(":", "_")
t1_date = t1.split("/")[0]
folder_name = f'data_{t1_date}_{t1_formatted}_to_{t2_formatted}'
os.makedirs(folder_name, exist_ok=True)

# Loop over each probe to load, process, and save data
for probe in probes:
    # Load various datasets from the MMS mission for the extended time range and probe
    pyspedas.mms.fsm(probe=probe, trange=[t1_ext, t2_ext], time_clip=True)
    pyspedas.mms.dsp(probe=probe, trange=[t1_ext, t2_ext], datatype=['epsd', 'bpsd'], data_rate='fast', level='l2', time_clip=True)
    pyspedas.mms.fgm(trange=[t1_ext, t2_ext], probe=probe, time_clip=True)
    pyspedas.mms.fpi(center_measurement=True, datatype=['des-moms'], trange=[t1_ext, t2_ext], probe=probe, time_clip=True)
    pyspedas.mms.mec(trange=[t1_ext, t2_ext], probe=probe, time_clip=True)

    # Retrieve the loaded data
    B = get_data(f'mms{probe}_fgm_b_gse_srvy_l2_btot', xarray=True)
    B_GSE = get_data(f'mms{probe}_fgm_b_gse_srvy_l2', xarray=True)
    density = get_data(f'mms{probe}_des_numberdensity_fast', xarray=True)
    pos_xr = get_data(f'mms{probe}_mec_r_gse', xarray=True)
    mlat_data = get_data('mms1_mec_mlat', xarray=True)
    l_dipole_data = get_data('mms1_mec_l_dipole', xarray=True)
    mlt_data = get_data('mms1_mec_mlt', xarray=True)
    Dst_data = get_data('mms1_mec_dst', xarray=True)
    Kp_data = get_data('mms1_mec_kp', xarray=True)

    # Convert time data from nanoseconds to datetime objects and ensure they are timezone-aware
    def convert_to_datetime(time_data):
        return pd.to_datetime(time_data, unit='ns', utc=True)

    common_time = convert_to_datetime(B.time.data)
    B_GSE_time = convert_to_datetime(B_GSE.time.data)
    density_time = convert_to_datetime(density.time.data)
    pos_xr_time = convert_to_datetime(pos_xr.time.data)
    mlat_time = convert_to_datetime(mlat_data.time.data)
    l_dipole_time = convert_to_datetime(l_dipole_data.time.data)
    mlt_time = convert_to_datetime(mlt_data.time.data)
    Dst_time = convert_to_datetime(Dst_data.time.data)
    Kp_time = convert_to_datetime(Kp_data.time.data)

    # Function to interpolate data to match a common time resolution
    def interpolate_to_common_time(original_time, original_data, common_time):
        common_time_numeric = common_time.astype(int) / 1e9  # Convert to seconds
        original_time_numeric = original_time.astype(int) / 1e9
        if original_data.ndim == 1:
            interpolated_data = np.interp(common_time_numeric, original_time_numeric, original_data)
        else:
            interpolated_data = np.array([np.interp(common_time_numeric, original_time_numeric, original_data[:, i]) for i in range(original_data.shape[1])]).T
        return interpolated_data

    # Interpolate B_GSE, density, and position data to match B's time resolution
    interpolated_B_GSE = interpolate_to_common_time(B_GSE_time, B_GSE.data, common_time)
    interpolated_density = interpolate_to_common_time(density_time, density.data, common_time)
    interpolated_pos = interpolate_to_common_time(pos_xr_time, pos_xr.data, common_time)
    interpolated_mlat = interpolate_to_common_time(mlat_time, mlat_data.data, common_time)
    interpolated_l_dipole = interpolate_to_common_time(l_dipole_time, l_dipole_data.data, common_time)
    interpolated_mlt = interpolate_to_common_time(mlt_time, mlt_data.data, common_time)
    interpolated_Dst = interpolate_to_common_time(Dst_time, Dst_data.data, common_time)
    interpolated_Kp = interpolate_to_common_time(Kp_time, Kp_data.data, common_time)


    # Filter the data for the specified time range (t1 to t2)
    mask = (common_time >= t1_aware) & (common_time <= t2_aware)
    filtered_common_time = common_time[mask]
    filtered_B = B.data[mask]
    filtered_B_GSE = interpolated_B_GSE[mask]
    filtered_density = interpolated_density[mask]
    filtered_pos = interpolated_pos[mask]
    filtered_mlat = interpolated_mlat[mask]
    filtered_l_dipole = interpolated_l_dipole[mask]
    filtered_mlt = interpolated_mlt[mask]
    filtered_Dst = interpolated_Dst[mask]
    filtered_Kp = interpolated_Kp[mask]

    # Convert filtered common time back to original string format with milliseconds
    original_time_format = filtered_common_time.strftime('%Y-%m-%d/%H:%M:%S.%f')

    # Calculate relative time in seconds from the start time
    relative_time_seconds = (filtered_common_time - filtered_common_time[0]).total_seconds()

    # Create a DataFrame with original time format and relative time seconds
    data = {
        'Time': original_time_format,
        'RelativeTime': relative_time_seconds,
        'Bt': filtered_B,
        'Bx': filtered_B_GSE[:, 0],
        'By': filtered_B_GSE[:, 1],
        'Bz': filtered_B_GSE[:, 2],
        'Density': filtered_density,
        'Rx': filtered_pos[:, 0],
        'Ry': filtered_pos[:, 1],
        'Rz': filtered_pos[:, 2],
        'L_Dipole' : filtered_l_dipole,
        'MLT' : filtered_mlt,
        'MLat': filtered_mlat,
        'Dst': filtered_Dst,
        'Kp' : filtered_Kp,
    }

    df = pd.DataFrame(data)

    # Construct the filename and save the CSV file in the created folder
    filename = f'{folder_name}/mms{probe}_data_{t1_date}_{t1_formatted}_to_{t2_formatted}.csv'
    df.to_csv(filename, index=False)

    # Print confirmation message with the saved filename
    print(f"Data saved to {filename}")


### Plotting the CSV files

In [ ]:
# Loop over each probe to load and plot data
for probe in probes:
    filename = f'{folder_name}/mms{probe}_data_{t1_date}_{t1_formatted}_to_{t2_formatted}.csv'
    
    # Load the data from CSV
    df = pd.read_csv(filename)

    # Convert the Time column to datetime
    df['Time'] = pd.to_datetime(df['Time'], format='%Y-%m-%d/%H:%M:%S.%f')

    # Plot the data
    plt.figure(figsize=(15, 10))
    plt.suptitle(f'MMS{probe} Data')

    # Plot B
    plt.subplot(3, 2, 1)
    plt.plot(df['Time'], df['Bt'])
    plt.title('Magnetic Field B')
    plt.xlabel('Time (HH:MM:SS)')
    plt.ylabel('Bt (nT)')
    plt.xticks(rotation=45)

    # Plot Bx
    plt.subplot(3, 2, 2)
    plt.plot(df['Time'], df['Bx'])
    plt.title('Magnetic Field Bx')
    plt.xlabel('Time (HH:MM:SS)')
    plt.ylabel('Bx (nT)')
    plt.xticks(rotation=45)

    # Plot By
    plt.subplot(3, 2, 3)
    plt.plot(df['Time'], df['By'])
    plt.title('Magnetic Field By')
    plt.xlabel('Time (HH:MM:SS)')
    plt.ylabel('By (nT)')
    plt.xticks(rotation=45)

    # Plot Bz
    plt.subplot(3, 2, 4)
    plt.plot(df['Time'], df['Bz'])
    plt.title('Magnetic Field Bz')
    plt.xlabel('Time (HH:MM:SS)')
    plt.ylabel('Bz (nT)')
    plt.xticks(rotation=45)

    # Plot Density
    plt.subplot(3, 2, 5)
    plt.plot(df['Time'], df['Density'])
    plt.title('Density')
    plt.xlabel('Time (HH:MM:SS)')
    plt.ylabel('Density (cm^-3)')
    plt.xticks(rotation=45)

    # Plot Position
    plt.subplot(3, 2, 6)
    plt.plot(df['Time'], df['Rx'], label='PosX')
    plt.plot(df['Time'], df['Ry'], label='PosY')
    plt.plot(df['Time'], df['Rz'], label='PosZ')
    plt.title('Position')
    plt.xlabel('Time (HH:MM:SS)')
    plt.ylabel('Position (km)')
    plt.legend()
    plt.xticks(rotation=45)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])  # Adjust layout to include suptitle
    plt.savefig('pdf.pdf')
    plt.show()


### Plotting data directly from probes

In [ ]:
for probe in probes:
    # Load data for the current probe
    pyspedas.mms.fsm(probe=probe, trange=[t1, t2], time_clip=True)
    pyspedas.mms.dsp(probe=probe, trange=[t1, t2], datatype=['epsd', 'bpsd'], data_rate='fast', level='l2', time_clip=True)
    pyspedas.mms.fgm(trange=[t1, t2], probe=probe, time_clip=True)
    pyspedas.mms.fpi(center_measurement=True, datatype=['des-moms'], trange=[t1, t2], probe=probe, time_clip=True)
    pyspedas.mms.mec(trange=[t1, t2], probe=probe, time_clip=True)
    
    tplot(f'mms{probe}_des_numberdensity_fast')
    options(f'mms{probe}_dsp_epsd_x', 'yrange', [40, 1000])
    options(f'mms{probe}_dsp_epsd_x', 'ylog', 0)
    tplot(f'mms{probe}_dsp_epsd_x')
    tplot(f'mms{probe}_fgm_b_gse_srvy_l2')
    tplot(f'mms{probe}_mec_r_gse')

# Data Aquisition

In [ ]:
import os
import pandas as pd
import numpy as np
from datetime import timedelta
import pyspedas
import xarray as xr
from pytplot import get_data

# Read time intervals from CSV
time_intervals_csv = 'time_interval_formatted_2016_3_1_to_7 copy.csv'
time_intervals = pd.read_csv(time_intervals_csv)

# List of probes to process
probes = [1]

# Helper functions
def convert_to_datetime(time_data):
    return pd.to_datetime(time_data, unit='ns', utc=True)

def interpolate_to_common_time(original_time, original_data, common_time):
    common_sec = common_time.astype(int) / 1e9
    original_sec = original_time.astype(int) / 1e9
    if original_data.ndim == 1:
        return np.interp(common_sec, original_sec, original_data)
    return np.array([
        np.interp(common_sec, original_sec, original_data[:, i])
        for i in range(original_data.shape[1])
    ]).T

# Main loop over time intervals
for _, row in time_intervals.iterrows():
    t1 = row['T1']
    t2 = row['T2']

    # Time extensions
    t1_ext = (pd.to_datetime(t1) - timedelta(minutes=5)).strftime('%Y-%m-%d/%H:%M:%S')
    t2_ext = (pd.to_datetime(t2) + timedelta(minutes=5)).strftime('%Y-%m-%d/%H:%M:%S')
    t1_date = t1.split("/")[0]
    t1_fmt = t1.split("/")[1].replace(":", "_")
    t2_fmt = t2.split("/")[1].replace(":", "_")
    folder_name = f'data_{t1_date}_{t1_fmt}_to_{t2_fmt}'
    os.makedirs(folder_name, exist_ok=True)

    t1_aware = pd.to_datetime(t1).tz_localize('UTC')
    t2_aware = pd.to_datetime(t2).tz_localize('UTC')

    for probe in probes:
        # Download data
        pyspedas.mms.fgm(probe=probe, trange=[t1_ext, t2_ext], time_clip=True)
        pyspedas.mms.fpi(probe=probe, datatype=['des-moms'], center_measurement=True, trange=[t1_ext, t2_ext], time_clip=True)
        pyspedas.mms.mec(probe=probe, trange=[t1_ext, t2_ext], time_clip=True)
        pyspedas.mms.dsp(probe=probe, trange=[t1_ext, t2_ext], datatype=['epsd'], data_rate='fast', level='l2', time_clip=True)

        # Retrieve data
        B = get_data(f'mms{probe}_fgm_b_gse_srvy_l2_btot', xarray=True)
        B_vec = get_data(f'mms{probe}_fgm_b_gse_srvy_l2', xarray=True)
        density = get_data(f'mms{probe}_des_numberdensity_fast', xarray=True)
        pos = get_data(f'mms{probe}_mec_r_gse', xarray=True)
        mlat = get_data(f'mms{probe}_mec_mlat', xarray=True)
        l_dipole = get_data(f'mms{probe}_mec_l_dipole', xarray=True)
        mlt = get_data(f'mms{probe}_mec_mlt', xarray=True)
        dst = get_data(f'mms{probe}_mec_dst', xarray=True)
        kp = get_data(f'mms{probe}_mec_kp', xarray=True)
        epsd = get_data(f'mms{probe}_dsp_epsd_x', xarray=True)

        if B is None or B_vec is None or density is None or pos is None:
            print(f"Missing core data for MMS{probe}, skipping.")
            continue

        common_time = convert_to_datetime(B.time.data)

        # Interpolate
        B_vec_interp = interpolate_to_common_time(convert_to_datetime(B_vec.time.data), B_vec.data, common_time)
        density_interp = interpolate_to_common_time(convert_to_datetime(density.time.data), density.data, common_time)
        pos_interp = interpolate_to_common_time(convert_to_datetime(pos.time.data), pos.data, common_time)
        mlat_interp = interpolate_to_common_time(convert_to_datetime(mlat.time.data), mlat.data, common_time)
        l_dipole_interp = interpolate_to_common_time(convert_to_datetime(l_dipole.time.data), l_dipole.data, common_time)
        mlt_interp = interpolate_to_common_time(convert_to_datetime(mlt.time.data), mlt.data, common_time)
        dst_interp = interpolate_to_common_time(convert_to_datetime(dst.time.data), dst.data, common_time)
        kp_interp = interpolate_to_common_time(convert_to_datetime(kp.time.data), kp.data, common_time)

        # EPSD interpolation
        if epsd is not None:
            epsd_time = convert_to_datetime(epsd.time.data)
            epsd_freq = epsd.v.data
            epsd_power = epsd.data
            epsd_interp = np.empty((len(common_time), len(epsd_freq)))
            for i in range(len(epsd_freq)):
                epsd_interp[:, i] = np.interp(
                    common_time.astype(int) / 1e9,
                    epsd_time.astype(int) / 1e9,
                    epsd_power[:, i]
                )
            # Filter to 50–800 Hz
            freq_mask = (epsd_freq >= 50.0) & (epsd_freq <= 800.0)
            epsd_freq = epsd_freq[freq_mask]
            epsd_interp = epsd_interp[:, freq_mask]
        else:
            print(f"No EPSD data for MMS{probe}")
            epsd_freq = np.array([])
            epsd_interp = np.empty((len(common_time), 0))

        # Time filtering
        mask = (common_time >= t1_aware) & (common_time <= t2_aware)
        filtered_time = common_time[mask]
        rel_time_sec = (filtered_time - filtered_time[0]).total_seconds()

        # Build Dataset
        ds = xr.Dataset(
            data_vars=dict(
                Bt=("time", B.data[mask]),
                Bx=("time", B_vec_interp[mask, 0]),
                By=("time", B_vec_interp[mask, 1]),
                Bz=("time", B_vec_interp[mask, 2]),
                Density=("time", density_interp[mask]),
                Rx=("time", pos_interp[mask, 0]),
                Ry=("time", pos_interp[mask, 1]),
                Rz=("time", pos_interp[mask, 2]),
                L_Dipole=("time", l_dipole_interp[mask]),
                MLT=("time", mlt_interp[mask]),
                MLat=("time", mlat_interp[mask]),
                Dst=("time", dst_interp[mask]),
                Kp=("time", kp_interp[mask]),
                EPSD=(["time", "frequency"], epsd_interp[mask])
            ),
            coords=dict(
                time=("time", filtered_time),
                frequency=("frequency", epsd_freq),
                RelativeTime=("time", rel_time_sec)
            ),
            attrs=dict(
                description=f"MMS{probe} full dataset including EPSD",
                source="NASA MMS via pySPEDAS",
            )
        )

        # Save to NetCDF
        out_file = os.path.join('DATA_Week', f"mms{probe}_multidimensional_{t1_date}_{t1_fmt}_to_{t2_fmt}.nc")
        ds.to_netcdf(out_file)
        print(f"Saved NetCDF: {'DATA_Week'}")
